## Cynthia Gao - HW 6

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/Users/cynthiagao/dso576/project/LA_listings.csv')
raw = df.copy()

df.head()

,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,2708,"Run Runyon, Beaut Furn Mirror Mini-Suit w/ Fir...",3008.0,1.462506e+18,⁨Chas.⁩,City of Los Angeles,Hollywood,34.09625,-118.34605,Private room,67.0,30.0,47,2026-05-26,0.32,2.0,301,3,NaN
1,2732,Zen Life at the Beach,3041.0,1.462506e+18,Yogapriestess,Other Cities,Santa Monica,34.00440,-118.48095,Private room,214.0,1.0,24,2022-08-21,0.13,2.0,350,0,228269
2,6033,Poolside Serenity Studio,11619.0,1.462507e+18,Sarah,City of Los Angeles,Woodland Hills,34.16887,-118.64478,Entire home/apt,100.0,30.0,19,2020-03-22,0.09,7.0,270,0,NaN
3,6931,"RUN Runyon, Beau Furn Rms Terrace Hollyw Hill ...",3008.0,1.462506e+18,⁨Chas.⁩,City of Los Angeles,Hollywood,34.09626,-118.34372,Private room,95.0,30.0,39,2025-12-15,0.19,2.0,259,2,NaN
4,7874,HD Wi-Fi E Bellflower Cerritos Lakewood LB LA,21700.0,1.462507e+18,Henry,Other Cities,Bellflower,33.87687,-118.11444,Private room,113.0,1.0,26,2026-04-02,0.29,4.0,103,9,NaN


In [3]:
df.shape

(43932, 19)

### 1. State the project vision and question (DONE)

Our project will create a dashboard for travelers to better understand the Los Angeles Airbnb market and help users make better decisions when choosing a neighborhood, room type, and price range by comparing value, demand, and availability. Our business question is: Which Los Angeles neighborhoods and room types offer travelers the best combination of value (price relative to comparable listings), demand (review activity), and availability, and how do these vary across price ranges?

### 2. Define one row  (DONE)

In [4]:
df.iloc[[0]]

,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,2708,"Run Runyon, Beaut Furn Mirror Mini-Suit w/ Fir...",3008.0,1.462506e+18,⁨Chas.⁩,City of Los Angeles,Hollywood,34.09625,-118.34605,Private room,67.0,30.0,47,2026-05-26,0.32,2.0,301,3,NaN


One row is one Airbnb listing in Los Angeles, each with a unique identifier in the column "id." 

### 3. Preserve the source (DONE)

Link: https://insideairbnb.com/get-the-data/
We are only using "listings.csv" from Los Angeles. The data is from June 15th, 2026 and we downloaded it on September 21st, 2026.

In [5]:
# Load the data with pandas
df = pd.read_csv('/Users/cynthiagao/dso576/project/LA_listings.csv') # File Path
raw = df.copy() # Save original data to raw variable

### 4. Inspect columns and type (DONE)

In [6]:
dtype_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isna().sum(),
    "n_unique": df.nunique(),
})
print(dtype_summary)

                                  dtype  nulls  n_unique
id                                int64      0     43932
name                             object      0     42715
host_id                         float64    181     21024
host_profile_id                 float64    395     20969
host_name                        object    384      8024
neighbourhood_group              object      0         3
neighbourhood                    object      0       265
latitude                        float64      0     33397
longitude                       float64      0     35509
room_type                        object      0         4
price                           float64   5570      2338
minimum_nights                  float64     11       100
number_of_reviews                 int64      0       707
last_review                      object  11311      2888
reviews_per_month               float64  11311       902
calculated_host_listings_count  float64    181        83
availability_365               

We need to change "last_review" to a datetime, but everything else looks good.

In [7]:
# Change last_review to datetime
df["last_review"] = pd.to_datetime(df["last_review"])

# Optional: label columns to category
for c in ["room_type", "neighbourhood_group", "neighbourhood"]:
    df[c] = df[c].astype("category")

### 5. Standardize Text (DONE)

In [8]:
INVISIBLE = r"[\u200b-\u200f\u202a-\u202e\u2060-\u2069\ufeff]"

def clean_text(s):
    return (s.str.replace(INVISIBLE, "", regex=True)   # hidden formatting characters
             .str.replace(r"\s+", " ", regex=True)     # spaces/tabs/newlines -> one space
             .str.strip())

text_cols = ["name", "host_name", "neighbourhood_group",
             "neighbourhood", "room_type", "license"]

for c in text_cols:
    df[c] = clean_text(df[c].astype("string"))

# Uppercase license codes only (leaves free-text licenses alone)
is_code = df["license"].str.fullmatch(r"[A-Za-z]{2,5}\d{2}-\d{3,6}", na=False)
df.loc[is_code, "license"] = df.loc[is_code, "license"].str.upper()

In [9]:
for c in ["name", "host_name", "license"]:
    changed = raw[c].fillna("") != df[c].fillna("")
    print(c, "rows changed:", changed.sum())
    print(pd.DataFrame({"before": raw.loc[changed, c].map(repr),
                        "after": df.loc[changed, c].map(repr)}).head(5))

name rows changed: 1070
                                                before  \
71   'private Space/Treehouse in a  mid century home.'   
79     'Silver Lake  million dollar view * luxe patio'   
84   'Glamorous Silver Lake hills Luxe\npatio city ...   
108  'Master Studio with Vaulted Ceiling  and jette...   
166              'Venice Sunset 3BDR,  Steps to Beach'   

                                                 after  
71    'private Space/Treehouse in a mid century home.'  
79      'Silver Lake million dollar view * luxe patio'  
84   'Glamorous Silver Lake hills Luxe patio city v...  
108  'Master Studio with Vaulted Ceiling and jetted...  
166               'Venice Sunset 3BDR, Steps to Beach'  
host_name rows changed: 257
                   before     after
0     '\u2068Chas.\u2069'   'Chas.'
3     '\u2068Chas.\u2069'   'Chas.'
282    '\u2068C.S.\u2069'    'C.S.'
315  '\u20682Great\u2069'  '2Great'
317  '\u20682Great\u2069'  '2Great'
license rows changed: 148
              b

### 6. Investigate duplicates (DONE)
There are no exact duplicates, but there are some near duplicates. However, I won't delete any of them because sometimes the same Airbnnb host can have multiple rooms available on the same property, so there does not seem to be an data errors here.

In [10]:
# 1. Exact duplicate rows: every column identical
print("Rows involved in exact duplicates:", df.duplicated(keep=False).sum())
print("Extra copies (would be removed):", df.duplicated().sum())

# 2. Repeated IDs: same listing id appearing more than once
id_dups = df["id"].duplicated(keep=False)
print("Rows with a repeated id:", id_dups.sum())
print("Distinct ids that repeat:", df.loc[id_dups, "id"].nunique())

# 3. Possible duplicate listings: different ids, same host, name, and location
key = ["host_id", "name", "latitude", "longitude"]
near_dups = df.duplicated(subset=key, keep=False) & df["host_id"].notna()
print("Rows in possible duplicate groups:", near_dups.sum())
print("Extra rows beyond the first in each group:",
      (df.duplicated(subset=key) & df["host_id"].notna()).sum())

# Look at the possible duplicates side by side
cols = ["id", "host_id", "name", "room_type", "price",
        "number_of_reviews", "last_review", "availability_365"]
print(df[near_dups].sort_values(key)[cols].head(20))

Rows involved in exact duplicates: 0
Extra copies (would be removed): 0
Rows with a repeated id: 0
Distinct ids that repeat: 0
Rows in possible duplicate groups: 224
Extra rows beyond the first in each group: 147
                        id     host_id  \
12562             52012376  13856090.0   
12814             52554329  13856090.0   
12819             52558861  13856090.0   
20387   933432935352409329  30850484.0   
22519  1045095351607895115  30850484.0   
15600   680301812424175952  30850484.0   
21263   980443156422212266  30850484.0   
21442   986191935520495605  30850484.0   
21867  1006581238129077339  30850484.0   
19912   924549795854323880  30850484.0   
20190   932609670481025747  30850484.0   
19917   924659728592147895  30850484.0   
20174   931315644606335685  30850484.0   
20709   960138862111782314  30850484.0   
20717   960168784282288278  30850484.0   
21236   990718205801306486  30850484.0   
21248   991319625339526456  30850484.0   
15599   680301689290798979  308

In [11]:
compare = df[near_dups].groupby(key).agg(
    n_listings=("id", "count"),
    n_room_types=("room_type", "nunique"),
    n_prices=("price", "nunique"),
    total_reviews=("number_of_reviews", "sum"),
)
print(compare.sort_values("n_listings", ascending=False).head(15))

                                                                                     n_listings  \
host_id     name                                              latitude  longitude                 
668954332.0 Cozy Private Room with Attached Bath              34.023830 -118.307960          19   
107434423.0 Blueground | Downtown, elevator, nr parks         34.068162 -118.228736          10   
668954332.0 Private Bed W/ Own Bath Near KTOWN                34.023466 -118.307213           8   
            Private Bed W/ Private Bath Near KTOWN            34.023466 -118.307213           7   
            Cozy Private Room & Bath                          34.024035 -118.306055           6   
            Luxurious Bed Private W/ Bath                     34.036220 -118.347520           5   
            Bright Private Room + Ensuite Bathroom            34.020762 -118.301539           5   
            Beautiful Private Bed W/ Private Bath             34.023955 -118.306037           5   
          

### 7. Handle missing data (DONE)
I only filled one column, "reviews_per_month", because every missing value is a listing with 0 reviews. The rest of the columns with missing values will only be flagged and not removed because each column contains valuable information and removing or filling them with the mean or median values would render the dataset useless or too small.

In [12]:
missing_before = df.isna().sum()
print(missing_before[missing_before > 0])

# Is the missing data structural (explained by other columns)?
print((df["reviews_per_month"].isna() == df["last_review"].isna()).all())                 # expect True
print((df.loc[df["reviews_per_month"].isna(), "number_of_reviews"] == 0).all())           # expect True
print((df["host_id"].isna() == df["calculated_host_listings_count"].isna()).all())        # expect True
print(df.loc[df["minimum_nights"].isna(), "availability_365"].describe())  

host_id                             181
host_profile_id                     395
host_name                           384
price                              5570
minimum_nights                       11
last_review                       11311
reviews_per_month                 11311
calculated_host_listings_count      181
license                           30740
dtype: int64
True
True
True
count    11.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: availability_365, dtype: float64


In [13]:
# Flags first, while the original missing values are still visible
df["has_price"]      = df["price"].notna()
df["never_reviewed"] = df["last_review"].isna()
df["host_missing"]   = df["host_id"].isna()
df["license_listed"] = df["license"].notna()

# The only fill: no reviews means 0 reviews per month
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

In [14]:
missing_after = df.isna().sum().reindex(missing_before.index)

report = pd.DataFrame({"before": missing_before, "after": missing_after})
report = report[report["before"] > 0].copy()
report["pct_of_rows"] = (report["before"] / len(df) * 100).round(1)

report["action"] = pd.Series({
    "price": "kept missing, flagged (has_price)",
    "last_review": "kept missing, flagged (never_reviewed)",
    "reviews_per_month": "filled with 0",
    "host_id": "kept missing, flagged (host_missing)",
    "host_profile_id": "kept missing",
    "host_name": "kept missing",
    "calculated_host_listings_count": "kept missing",
    "minimum_nights": "kept missing",
    "license": "kept missing, flagged (license_listed)",
})
print(report)

                                before  after  pct_of_rows  \
host_id                            181    181          0.4   
host_profile_id                    395    395          0.9   
host_name                          384    384          0.9   
price                             5570   5570         12.7   
minimum_nights                      11     11          0.0   
last_review                      11311  11311         25.7   
reviews_per_month                11311      0         25.7   
calculated_host_listings_count     181    181          0.4   
license                          30740  30740         70.0   

                                                                action  
host_id                           kept missing, flagged (host_missing)  
host_profile_id                                           kept missing  
host_name                                                 kept missing  
price                                kept missing, flagged (has_price)  
minimum_nights

### 8. Check suspicious values (DONE)
My check flagged 34 potential impossible values where the number of reviews in the last 12 months was more than 1 but the last review date was over a year ago. I retained these values and flagged them because these rows may still contain useful information about price, location, room type, and availability. 

In [15]:
snapshot = df["last_review"].max()          # proxy for the scrape date
print("Earliest / latest review:", df["last_review"].min(), snapshot)

print("Future dates:", (df["last_review"] > pd.Timestamp.today()).sum())
print("Before Airbnb existed (2008):", (df["last_review"] < "2008-08-01").sum())

Earliest / latest review: 2011-05-21 00:00:00 2026-06-22 00:00:00
Future dates: 0
Before Airbnb existed (2008): 0


In [16]:
impossible = {
    "price <= 0":                       (df["price"] <= 0).sum(),
    "minimum_nights < 1":               (df["minimum_nights"] < 1).sum(),
    "availability outside 0-365":       ((df["availability_365"] < 0) | (df["availability_365"] > 365)).sum(),
    "negative review counts":           ((df["number_of_reviews"] < 0) | (df["number_of_reviews_ltm"] < 0)).sum(),
    "reviews_ltm > total reviews":      (df["number_of_reviews_ltm"] > df["number_of_reviews"]).sum(),
    "reviews > 0 but no last_review":   ((df["number_of_reviews"] > 0) & df["last_review"].isna()).sum(),
    "reviews == 0 but has last_review": ((df["number_of_reviews"] == 0) & df["last_review"].notna()).sum(),
    "ltm > 0 but last review > 1 yr old": ((df["number_of_reviews_ltm"] > 0) &
                                           (df["last_review"] < snapshot - pd.Timedelta(days=366))).sum(),
    "coordinates outside LA County":    (~df["latitude"].between(33.2, 34.9) |
                                         ~df["longitude"].between(-119.0, -117.5)).sum(),
}
print(pd.Series(impossible))

price <= 0                             0
minimum_nights < 1                     0
availability outside 0-365             0
negative review counts                 0
reviews_ltm > total reviews            0
reviews > 0 but no last_review         0
reviews == 0 but has last_review       0
ltm > 0 but last review > 1 yr old    34
coordinates outside LA County          0
dtype: int64


In [17]:
cols = ["id", "neighbourhood", "room_type", "price", "minimum_nights",
        "number_of_reviews", "reviews_per_month", "availability_365"]

# Price extremes within each room type
p01 = df.groupby("room_type")["price"].transform(lambda s: s.quantile(0.01))
p99 = df.groupby("room_type")["price"].transform(lambda s: s.quantile(0.99))
df["flag_price_high"] = df["price"] > p99
df["flag_price_low"]  = df["price"] < p01

# Stays that make short-term booking impossible
df["flag_min_nights_gt_365"] = df["minimum_nights"] > 365

# Extreme review activity
df["flag_reviews_extreme"] = df["reviews_per_month"] > df["reviews_per_month"].quantile(0.999)

# Host portfolio size vs. what the file actually shows
actual_count = df.groupby("host_id")["id"].transform("count")
df["flag_host_count_mismatch"] = (actual_count != df["calculated_host_listings_count"]) & df["host_id"].notna()

# Look at the extreme records yourself
print(df.nlargest(15, "price")[cols])
print(df.nsmallest(15, "price")[cols])
print(df.nlargest(10, "minimum_nights")[cols])
print(df.nlargest(10, "number_of_reviews")[cols])

# Licenses reused across many listings
print(df["license"].value_counts().head(10))

# Summary of all flags
flag_cols = [c for c in df.columns if c.startswith("flag_")]
print(df[flag_cols].sum())
print("Rows with at least one flag:", df[flag_cols].any(axis=1).sum())

                        id         neighbourhood        room_type    price  \
5978              25870842         Beverly Hills  Entire home/apt  20583.0   
4932              20471072               Bel-Air  Entire home/apt  18396.0   
9894              43302796         Beverly Hills  Entire home/apt  17612.0   
22776  1061241264477237025                Malibu  Entire home/apt  16389.0   
37102  1569397151095732791                Malibu  Entire home/apt  15490.0   
27418  1257476611381419431  Hollywood Hills West  Entire home/apt  14620.0   
20403   939012920225974975  Hollywood Hills West  Entire home/apt  14539.0   
24353  1136418613533078238               Bel-Air  Entire home/apt  14084.0   
30122  1363305939461956001         Beverly Hills  Entire home/apt  14037.0   
23338  1079908245408870997                Malibu  Entire home/apt  13816.0   
40701  1648267330829565088                Malibu  Entire home/apt  13500.0   
36939  1564859559451192704                Malibu  Entire home/ap

### 9. Verify merges (NOT APPLICABLE)
We are only working with one table for LA Airbnb listings.

### 10. Reconcile changes (DONE)
Cleaning did not change our main totals: we started and ended with 43,932 listings, no rows were removed, and no tables were merged, so listing counts, price statistics, and review totals are unchanged. Only a few values moved because of standardization: distinct license values fell from 7,718 to 7,691 (the same permit in different capitalizations), distinct listing names dropped slightly (extra spaces and hidden characters removed), and mean reviews per month fell because 11,311 never-reviewed listings were assigned their true rate of 0.

In [18]:
start_rows = len(raw)
final_rows = len(df)
print("Starting rows:", start_rows)
print("Final rows:   ", final_rows)
print("Difference:   ", start_rows - final_rows)
print("Columns before/after:", raw.shape[1], "/", df.shape[1])   # flags add columns, not rows

# Which listings, if any, are missing from the cleaned data?
raw_ids = set(raw["id"].astype(str))
df_ids  = set(df["id"].astype(str))
removed = raw_ids - df_ids
added   = df_ids - raw_ids
print("IDs removed:", len(removed))
print("IDs added:  ", len(added))

# The accounting has to balance
assert start_rows - len(removed) + len(added) == final_rows, "Row counts don't reconcile"

Starting rows: 43932
Final rows:    43932
Difference:    0
Columns before/after: 19 / 28
IDs removed: 0
IDs added:   0


In [19]:
def totals(d):
    return pd.Series({
        "rows": len(d),
        "distinct listing ids": d["id"].nunique(),
        "distinct hosts": d["host_id"].nunique(),
        "listings with a price": d["price"].notna().sum(),
        "median price": d["price"].median(),
        "mean price": d["price"].mean(),
        "total reviews": d["number_of_reviews"].sum(),
        "total reviews, last 12 months": d["number_of_reviews_ltm"].sum(),
        "mean reviews_per_month": d["reviews_per_month"].mean(),
        "share with min nights >= 30": (d["minimum_nights"].dropna() >= 30).mean(),
        "listings with a license": d["license"].notna().sum(),
        "distinct license values": d["license"].nunique(),
        "distinct listing names": d["name"].nunique(),
    })

comparison = pd.DataFrame({"raw": totals(raw), "cleaned": totals(df)})
comparison["change"] = comparison["cleaned"] - comparison["raw"]
print(comparison.round(2))

                                      raw     cleaned  change
rows                             43932.00    43932.00    0.00
distinct listing ids             43932.00    43932.00    0.00
distinct hosts                   21024.00    21024.00    0.00
listings with a price            38362.00    38362.00    0.00
median price                       224.00      224.00    0.00
mean price                         383.14      383.14    0.00
total reviews                  1856294.00  1856294.00    0.00
total reviews, last 12 months   377493.00   377493.00    0.00
mean reviews_per_month               1.39        1.03   -0.36
share with min nights >= 30          0.47        0.47    0.00
listings with a license          13192.00    13192.00    0.00
distinct license values           7718.00     7691.00  -27.00
distinct listing names           42715.00    42707.00   -8.00


### 11. Check calculations (DONE)
- Totals (sums and counts): sum() skips missing values, so the total covers only the rows that have a value. It is not an estimate for the full dataset.
- Averages and medians: mean() and median() skip missing values, so the denominator is the number of non-missing rows, not the total row count. If rows are filled with 0 first, they enter the denominator and pull the average down.
- Shares and percentages: this is the easiest place to make a mistake. A comparison like df["minimum_nights"] >= 30 returns False for a missing value, so .mean() on it counts those rows in the denominator as "not 30+". Drop the missing rows first so the denominator is correct.
- Group results: each group's average rests on a different number of records. Report the count next to every average, because a neighborhood average built on 8 priced listings is not comparable to one built on 800.

In [20]:
metrics = {
    "Total listings":                       (len(df), 0),
    "Median/mean price":                    (df["price"].notna().sum(), df["price"].isna().sum()),
    "Mean reviews/month (all listings)":    (len(df), 0),
    "Mean reviews/month (reviewed only)":   ((~df["never_reviewed"]).sum(), df["never_reviewed"].sum()),
    "Share min nights >= 30":               (df["minimum_nights"].notna().sum(), df["minimum_nights"].isna().sum()),
    "Share from multi-listing hosts":       (df["calculated_host_listings_count"].notna().sum(),
                                             df["calculated_host_listings_count"].isna().sum()),
    "Share with a license":                 (len(df), 0),
}
report = pd.DataFrame(metrics, index=["records_used", "records_excluded"]).T
print(report)

# Compute the shares with the right denominators
print((df["minimum_nights"].dropna() >= 30).mean())
print((df["calculated_host_listings_count"].dropna() > 1).mean())
print(df["license"].notna().mean())

                                    records_used  records_excluded
Total listings                             43932                 0
Median/mean price                          38362              5570
Mean reviews/month (all listings)          43932                 0
Mean reviews/month (reviewed only)         32621             11311
Share min nights >= 30                     43921                11
Share from multi-listing hosts             43751               181
Share with a license                       43932                 0
0.46631451925047246
0.6505908436378597
0.30028225439315304


In [21]:
print(df[["neighbourhood_group", "neighbourhood", "room_type", "host_id"]].isna().sum())

neighbourhood_group      0
neighbourhood            0
room_type                0
host_id                181
dtype: int64


In [22]:
print(df.groupby("room_type", dropna=False)["price"].agg(
    listings="size",       # all rows in the group
    priced="count",        # rows with a non-missing price
    median_price="median"
))

                 listings  priced  median_price
room_type                                      
Entire home/apt     32454   28437         282.0
Hotel room            415     400         319.0
Private room        10771    9245          86.0
Shared room           292     280          37.0


### 12. Verify individual records (DONE)

In [23]:
def show(v):
    return "<missing>" if pd.isna(v) else repr(v)

def same(a, b):
    if pd.isna(a) and pd.isna(b):
        return True
    if pd.isna(a) or pd.isna(b):
        return False
    return a == b

# Pick records C, D, and E by condition (first matching row in the original)
idx_zero   = raw.index[raw["license"].str.fullmatch(r"0\d+", na=False)][0]
idx_unrev  = raw.index[raw["number_of_reviews"] == 0][0]
idx_nohost = raw.index[raw["host_id"].isna()][0]

# (label, index, column in df, EXPECTED value, column in raw to show as original)
checks = [
    ("A hidden chars + accent", 10883,      "host_name",         "Ray Pérez",                   "host_name"),
    ("B license capitalization", 2320,      "license",           "HSR19-005325",                "license"),
    ("C leading zero",           idx_zero,  "license",           raw.loc[idx_zero, "license"],  "license"),
    ("D never reviewed",         idx_unrev, "reviews_per_month", 0,                             "reviews_per_month"),
    ("D never reviewed",         idx_unrev, "last_review",       pd.NaT,                        "last_review"),
    ("E unknown host",           idx_nohost,"host_id",           pd.NA,                         "host_id"),
    ("E unknown host",           idx_nohost,"host_missing",      True,                          "host_id"),
]

rows = []
for label, idx, col, expected, raw_col in checks:
    original = raw.loc[idx, raw_col]
    actual = df.loc[idx, col]
    rows.append({
        "record": label, "index": idx, "column": col,
        "original": show(original), "expected": show(expected),
        "actual": show(actual), "match": same(expected, actual),
    })

pd.set_option("display.max_colwidth", None)
print(pd.DataFrame(rows).to_string(index=False))

                  record  index            column                      original       expected          actual  match
 A hidden chars + accent  10883         host_name '\u2068Ray Pérez\u202f\u2069'    'Ray Pérez'     'Ray Pérez'   True
B license capitalization   2320           license                'Hsr19-005325' 'HSR19-005325'  'HSR19-005325'   True
          C leading zero   2162           license                      '093189'       '093189'        '093189'   True
        D never reviewed     46 reviews_per_month                     <missing>              0 np.float64(0.0)   True
        D never reviewed     46       last_review                     <missing>      <missing>       <missing>   True
          E unknown host  22422           host_id                     <missing>      <missing>       <missing>   True
          E unknown host  22422      host_missing                     <missing>           True        np.True_   True


### 13. Test reproducability (DONE)
This pdf file was downloaded directly from the ipynb file, so upon re-running the entire script, everything is confirmed working.

### 14. Document remaining limitations (DONE)
Our main limitations are that 5,570 listings (12.7%) have no price and that we only have one snapshot of one city, so price comparisons rest on 87% of listings and cannot show changes over time, which would require the archived quarterly snapshots. Review counts are only an estimate for demand and there are no rating scores, so neighborhoods with fewer reviews may look weaker without actually being less popular, and we would need real booking data and review scores. Several unusual records were flagged instead of removed (extreme prices, 24 listings with minimum stays over 365 nights, 34 with conflicting review dates, and about 147 possible duplicates), and the license field is blank for 70% of listings with no way to tell "unlicensed" from "not reported," so averages could be skewed and any compliance conclusion would need verification.